# How a classical test works: the t-test

Somewhere out there are two **populations** — say, outcomes under treatment A and
under treatment B. Each is a whole distribution of individuals (the blue and orange
curves below). The truth has knobs we never observe directly: the **true difference
of the means** (Δ) and the **true variance** (σ²).

What we *do* observe is a **sample**: n points from each population — the dots.

The classical recipe goes:

1. State the **null hypothesis H₀**: *the two populations have the same mean*
   (Δ = 0) — against the **alternative H₁**: *they don't*.
2. Condense the entire sample into a single number, the **test statistic**

   $$t \;=\; \frac{\bar x_B - \bar x_A}{s_p \sqrt{2/n}}$$

   — the observed difference of sample means, measured in units of its own
   estimation noise.
3. *If H₀ were true*, t would follow a known distribution: **Student's t** (the dark
   curve). Notice that this is a statement about the *statistic*, not about the data —
   it is a different curve with its own shape (compare it with the pale gaussian).
4. The **p-value** is the probability, *under H₀*, of a t at least as extreme as the
   one you got: the shaded two-sided **tail area**.
5. If p ≤ **α** (fixed here at 0.05 — the small ticks on the baseline), call the result
   *significant* and **reject H₀**.

A map of the picture:

| concept | where to see it below |
|---|---|
| population | the two smooth curves — never observed directly |
| sample | the dots just above (blue) and below (orange) the line |
| H₀ | "blue and orange coincide" — drag Δ to 0 to *make it true* |
| test statistic | the marker labelled *t* on the second baseline |
| its null distribution | the dark Student-t curve — *not* the data's gaussian |
| p-value | the shaded tail area, also in numbers |
| α | the bar p must clear; ticks mark the critical t |


In [6]:
# The machinery of the demo. Pure functions: parameters -> SVG / HTML strings;
# the only state is the pool of random draws, kept fixed while you drag.
# Every panel here and in notebook 03 is a square box (aspect 1) on ONE scale:
# SC px per x-unit, K px per density-unit — so the panels compare directly.
import numpy as np
from scipy import stats

# blue = group A, orange = group B — same palette as the rest of the series
BLUE, ORANGE = "#3D6FB0", "#E0883A"
INK, MUTED, RULE, PALE = "#333333", "#888888", "#DDDDDD", "#BFBFBF"
TAIL = "#C96A5E"
FONT = "font-family:system-ui,-apple-system,sans-serif;"

POOL = 100
ALPHA = 0.05

# --- the shared square + scale (identical in notebook 03) -------------------
S = 350                            # side of the square plot box
HALF = 5.0                         # x half-range, in data / statistic units
YMAX = 1.25                        # y range, in units of probability density
SC = S / (2 * HALF)                # px per unit on x   (= 35.0)
K = S / YMAX                       # px per density unit on y  (= 280)
ML, MR, BH = 16, 16, 24            # left / right / below-baseline margins
FW = ML + S + MR                   # figure width
X0 = ML + S / 2                    # x = 0 at the centre of the box


def x_to_px(x):
    return X0 + x * SC


def y_to_px(base, dens, cap=S):
    """Density -> y, clipped so nothing climbs past `cap` pixels above base."""
    return base - min(dens * K, cap)


def make_pool(seed=None):
    rng = np.random.default_rng(seed)
    return dict(z1=rng.standard_normal(POOL), z2=rng.standard_normal(POOL),
                j1=rng.random(POOL), j2=rng.random(POOL))


def filled_curve(xs, dens, base, stroke, fill, op, sw=2):
    """A light-filled, top-truncated curve: fill polygon + stroke polyline."""
    pts = [(x_to_px(x), y_to_px(base, d)) for x, d in zip(xs, dens)]
    line = " ".join(f"{px:.1f},{py:.1f}" for px, py in pts)
    poly = f"{pts[0][0]:.1f},{base:.1f} {line} {pts[-1][0]:.1f},{base:.1f}"
    return (f"<polygon points='{poly}' fill='{fill}' opacity='{op}'/>"
            f"<polyline points='{line}' fill='none' stroke='{stroke}' stroke-width='{sw}'/>")


# ----------------------------------------------------------------------------
# 1 · the two populations and the sample in hand
# ----------------------------------------------------------------------------

def population_svg(delta, var, n, pool):
    sigma = var ** 0.5
    top = 14
    base = top + S
    height = base + BH
    xs = np.linspace(-HALF, HALF, 201)

    parts = []
    for mu, col in ((-delta / 2, BLUE), (delta / 2, ORANGE)):
        pdf = np.exp(-0.5 * ((xs - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))
        parts.append(filled_curve(xs, pdf, base, col, col, 0.10))

    parts.append(f"<line x1='{ML}' x2='{ML + S}' y1='{base}' y2='{base}' stroke='{RULE}' stroke-width='1'/>")

    for z, j, mu, col, side in ((pool["z1"], pool["j1"], -delta / 2, BLUE, -1),
                                (pool["z2"], pool["j2"], delta / 2, ORANGE, 1)):
        for zi, ji in zip(z[:n], j[:n]):
            x = max(-HALF, min(HALF, mu + sigma * zi))
            y = base + side * (9 + ji * 6)
            parts.append(f"<circle cx='{x_to_px(x):.1f}' cy='{y:.1f}' r='2.6' "
                         f"fill='{col}' opacity='0.75'/>")

    return (f"<svg width='{FW}' height='{height}' viewBox='0 0 {FW} {height}' "
            f"xmlns='http://www.w3.org/2000/svg'>" + "".join(parts) + "</svg>")


# ----------------------------------------------------------------------------
# 2 · the test statistic, its null distribution, and the p-value
# ----------------------------------------------------------------------------

def tstat_svg(t, p, df):
    top = 14
    base = top + S
    height = base + BH
    xs = np.linspace(-HALF, HALF, 241)

    parts = [filled_curve(xs, stats.t.pdf(xs, df), base, INK, INK, 0.05)]  # the t curve, lightly filled

    # two-sided tail area = the p-value, drawn on top of the fill
    tt = min(abs(t), HALF)
    for sign in (1, -1):
        xt = np.linspace(tt, HALF, 61) * sign
        seg = " ".join(f"{x_to_px(x):.1f},{y_to_px(base, q):.1f}"
                       for x, q in zip(xt, stats.t.pdf(xt, df)))
        parts.append(f"<polygon points='{x_to_px(sign * tt):.1f},{base} {seg} "
                     f"{x_to_px(sign * HALF):.1f},{base}' fill='{TAIL}' opacity='0.85'/>")

    # pale gaussian — a reference line (no fill), to show t ≠ normal
    pts_n = " ".join(f"{x_to_px(x):.1f},{y_to_px(base, q):.1f}" for x, q in zip(xs, stats.norm.pdf(xs)))
    parts.append(f"<polyline points='{pts_n}' fill='none' stroke='{PALE}' stroke-width='1.6'/>")
    # redraw the t stroke so it sits above the tail fill
    pts_t = " ".join(f"{x_to_px(x):.1f},{y_to_px(base, q):.1f}" for x, q in zip(xs, stats.t.pdf(xs, df)))
    parts.append(f"<polyline points='{pts_t}' fill='none' stroke='{INK}' stroke-width='2'/>")

    parts.append(f"<line x1='{ML}' x2='{ML + S}' y1='{base}' y2='{base}' stroke='{RULE}' stroke-width='1'/>")

    # critical values: the points in the tails where the tail area equals alpha
    tc = stats.t.ppf(1 - ALPHA / 2, df)
    if tc < HALF:
        ytop = min(y_to_px(base, stats.t.pdf(tc, df)), base - 52)
        for sign in (1, -1):
            parts.append(f"<line x1='{x_to_px(sign * tc):.1f}' x2='{x_to_px(sign * tc):.1f}' "
                         f"y1='{base}' y2='{ytop:.1f}' stroke='{MUTED}' stroke-width='1.3' "
                         f"stroke-dasharray='2 3'/>")
            parts.append(f"<text x='{x_to_px(sign * tc):.1f}' y='{base + 15:.0f}' text-anchor='middle' "
                         f"style='{FONT}' font-size='10' fill='{MUTED}'>p = α</text>")

    # observed t — a vertical line with a label kept inside the box
    xm = max(-HALF + 0.05, min(HALF - 0.05, t))
    lblx = min(max(x_to_px(xm), ML + 34), ML + S - 34)
    parts.append(f"<line x1='{x_to_px(xm):.1f}' x2='{x_to_px(xm):.1f}' y1='{base}' y2='{base - 168:.0f}' "
                 f"stroke='{INK}' stroke-width='1.4'/>")
    anchor = "start" if t < 0 else "end"
    dx = 6 if t < 0 else -6
    parts.append(f"<text x='{lblx + dx:.1f}' y='{base - 172:.0f}' text-anchor='{anchor}' style='{FONT}' "
                 f"font-size='12' font-weight='600' fill='{INK}'>t = {t:.2f}</text>")

    # p-value, top-left inside the box
    ptxt = f"p = {p:.3f}" if p >= 0.001 else "p &lt; 0.001"
    parts.append(f"<text x='{ML + 6}' y='{top + 22}' style='{FONT}' font-size='17' "
                 f"font-weight='700' fill='{TAIL}'>{ptxt}</text>")
    parts.append(f"<text x='{ML + 6}' y='{top + 38}' style='{FONT}' font-size='10' "
                 f"fill='{MUTED}'>two-sided tail area</text>")

    # legend, top-right inside the box
    lx = ML + S - 184
    parts.append(f"<line x1='{lx}' x2='{lx + 20}' y1='{top + 16}' y2='{top + 16}' stroke='{INK}' stroke-width='2'/>")
    parts.append(f"<text x='{lx + 26}' y='{top + 20}' style='{FONT}' font-size='11' fill='{INK}'>"
                 f"t under H₀ (df = {df})</text>")
    parts.append(f"<line x1='{lx}' x2='{lx + 20}' y1='{top + 32}' y2='{top + 32}' stroke='{PALE}' stroke-width='1.6'/>")
    parts.append(f"<text x='{lx + 26}' y='{top + 36}' style='{FONT}' font-size='11' fill='{MUTED}'>"
                 f"gaussian, for comparison</text>")

    return (f"<svg width='{FW}' height='{height}' viewBox='0 0 {FW} {height}' "
            f"xmlns='http://www.w3.org/2000/svg'>" + "".join(parts) + "</svg>")


def verdict_html(p):
    if p <= ALPHA:
        bg, txt = "#C0392B", "null hypothesis rejected"
    else:
        bg, txt = "#9AA0A6", "null hypothesis not rejected"
    return (f"<div style='{FONT}margin:6px 0 0 {ML}px'>"
            f"<span style='background:{bg};color:white;padding:5px 14px;border-radius:14px;"
            f"font-size:12.5px;font-weight:600;letter-spacing:.5px'>{txt}</span>"
            f"<span style='font-size:12px;color:{MUTED};margin-left:10px'>α = {ALPHA}</span></div>")

In [7]:
import ipywidgets as W

def slider(value, mn, mx, step, desc, cls=W.FloatSlider, **kw):
    return cls(value=value, min=mn, max=mx, step=step, description=desc,
               continuous_update=True, style={"description_width": "170px"},
               layout=W.Layout(width="430px"), **kw)

delta_s = slider(0.8, 0.0, 3.0, 0.05, "true difference · Δ", readout_format=".2f")
var_s   = slider(1.0, 0.25, 4.0, 0.05, "true variance · σ²", readout_format=".2f")
n_s     = slider(12, 3, 100, 1, "sample size · n per group", cls=W.IntSlider)

new_btn = W.Button(description="↻  new sample",
                   layout=W.Layout(width="150px", margin="4px 0 0 174px"))

pool = make_pool(seed=24)           # one fixed draw — dots don't jitter as you drag
pop_w, t_w, verdict_w = W.HTML(), W.HTML(), W.HTML()

def refresh(change=None):
    delta, var, n = delta_s.value, var_s.value, n_s.value
    sigma = var ** 0.5
    xa = -delta / 2 + sigma * pool["z1"][:n]
    xb =  delta / 2 + sigma * pool["z2"][:n]
    res = stats.ttest_ind(xa, xb)
    pop_w.value     = population_svg(delta, var, n, pool)
    t_w.value       = tstat_svg(res.statistic, res.pvalue, 2 * n - 2)
    verdict_w.value = verdict_html(res.pvalue)

def resample(_):
    pool.update(make_pool())        # only the button re-randomizes
    refresh()

for s in (delta_s, var_s, n_s):
    s.observe(refresh, "value")
new_btn.on_click(resample)
refresh()

def caption(txt):
    return W.HTML(f"<div style='{FONT}font-size:11px;letter-spacing:1.5px;"
                  f"color:{MUTED};text-transform:uppercase;margin:22px 0 2px'>{txt}</div>")

hint = W.HTML(f"<div style='{FONT}font-size:12px;color:{MUTED};font-style:italic;"
              "margin-bottom:4px'>drag a slider — the truth changes; "
              "press the button — a new sample is drawn</div>")

W.VBox([hint, delta_s, var_s, n_s, new_btn,
        caption("1 · two populations, one sample"), pop_w,
        caption("2 · the test statistic and its p-value"), t_w, verdict_w])


## Try this

1. **Drag Δ to 0.** H₀ is now *true*. Press **new sample** a dozen times: t dances
   around zero and the p-value bounces all over the place. About one press in twenty
   will dip below 0.05 and "discover" an effect that does not exist. That 1-in-20 is
   exactly α — the false-positive rate you signed up for.
2. **Δ = 0.5, n = 12**: a real but modest effect, rarely significant — the test is
   *underpowered*. Now drag n to 100: same truth, yet significance almost every time.
   t grows like √n, so *any* nonzero difference becomes "significant" with enough data.
   Significance measures evidence, not importance.
3. **Crank σ² up** and watch significance melt away. The t statistic is a tug of war:
   signal (Δ) against noise (σ, tamed by √n).
4. **Drop n to 3 or 4** and compare the dark curve with the pale gaussian: fat tails.
   With so little data, large t happen easily by chance, so the test demands a bigger t
   before it is surprised. By n ≈ 30 the two curves are indistinguishable.

### The moral

* The p-value answers exactly one question: *"**If** there were truly no difference,
  how surprising would my t be?"* It is a tail probability computed under H₀ —
  nothing more.
* It is **not** the probability that H₀ is true, and a rejection is **not** a measure
  of how big or important the effect is.
* "Reject H₀" is simply the machine stamping **positive** on this study. How much
  *belief* such a stamp deserves is a different question entirely — it is the subject
  of the next notebook.
